<a href="https://colab.research.google.com/github/SenTier1107/Appprogramming_2026/blob/main/NaverWebtoon_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Week 5 과제: 네이버 웹툰 크롤링

| 항목 | 내용 |
|------|------|
| **과목** | Web Crawling |
| **날짜** | 2026.04.10 |
| **대상 URL** | https://comic.naver.com/webtoon |

---

## 과제 목표

> 네이버 웹툰 메인 페이지에서 **"요일전체"** 아래 **이달의 신규 웹툰**의 **제목, 작가, 내용**을 텍스트로 스크래핑한다.

---

## 1. 왜 Selenium(동적 크롤링)을 사용하는가?

네이버 웹툰은 **React 기반 SPA(Single Page Application)** 입니다.

| 항목 | BeautifulSoup (정적) | Selenium (동적) |
|------|---------------------|----------------|
| 속도 | 빠름 | 느림 |
| JS 실행 | ❌ | ✅ |
| 리소스 | 가벼움 | 무거움 (브라우저 필요) |
| 적합한 사이트 | 전통적 HTML 사이트 | React/Vue/JS 기반 사이트 |

- **정적 크롤링 결과**: `None` ← JS 실행 전이라 빈 껍데기
- **동적 크롤링 결과**: 실제 웹툰 목록 데이터 ← JS 실행 후 채워진 내용

따라서 이 과제에서는 **Selenium**을 사용합니다.

---
## 2. 환경 설정

Colab 환경에서 Selenium을 사용하려면 Chrome 브라우저와 ChromeDriver를 설치해야 합니다.

`google-colab-selenium` 패키지를 사용하면 Colab에 최적화된 설정을 자동으로 해줍니다.

> ⚠️ 만약 방법 A가 안 되면, **방법 B**의 주석을 해제하고 사용하세요.

In [ ]:
# ── 방법 A: google-colab-selenium 패키지 (권장) ──
!pip install google-colab-selenium

---
## 3. 라이브러리 임포트

크롤링에 필요한 모듈들을 불러옵니다.

In [1]:
from selenium.webdriver.common.by import By             # 요소 탐색 방법 (CSS, XPATH 등)
from selenium.webdriver.support.ui import WebDriverWait         # 명시적 대기
from selenium.webdriver.support import expected_conditions as EC # 대기 조건 설정
from bs4 import BeautifulSoup                           # HTML 파싱 보조용
import time

print("✅ 라이브러리 임포트 완료")

✅ 라이브러리 임포트 완료


---
## 4. Chrome 브라우저 실행

Colab에는 화면(GUI)이 없으므로 **headless 모드**로 Chrome을 실행해야 합니다.

`google-colab-selenium`은 다음 옵션들을 **자동**으로 설정해줍니다:
- `--headless` : 화면 없이 백그라운드에서 브라우저 실행
- `--no-sandbox` : Colab 보안 환경 호환
- `--disable-dev-shm-usage` : 메모리 부족 방지

> 방법 A 실행 시 에러가 나면 → 방법 B의 주석을 해제하고 방법 A를 주석 처리하세요.

In [2]:
# ── 방법 A: google-colab-selenium 사용 (권장) ──
import google_colab_selenium as gs
driver = gs.Chrome()


print("✅ Chrome 브라우저 실행 완료")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Chrome 브라우저 실행 완료


---
## 5. 네이버 웹툰 페이지 접속

설정한 브라우저로 네이버 웹툰 페이지에 접속합니다.

```python
driver.get(url)  # 실제 브라우저가 해당 URL을 열고 JavaScript를 실행
```

In [3]:
# 네이버 웹툰 메인 페이지 접속
url = "https://comic.naver.com/webtoon"
driver.get(url)

print(f"접속 URL : {url}")
print(f"페이지 제목 : {driver.title}")

접속 URL : https://comic.naver.com/webtoon
페이지 제목 : 요일전체 : 네이버 웹툰


---
## 6. 페이지 로딩 대기

네이버 웹툰은 React SPA이므로, 페이지 접속 후 **JavaScript가 콘텐츠를 렌더링할 시간**이 필요합니다.

두 가지 대기 방법:
- `time.sleep(초)` : 무조건 지정 시간만큼 대기 (단순하지만 비효율적)
- `WebDriverWait` : 특정 요소가 나타날 때까지 대기 (더 안정적)

여기서는 `WebDriverWait`을 우선 사용하고, 실패 시 `time.sleep`으로 대체합니다.

In [4]:
# 페이지가 완전히 로딩될 때까지 대기
# React SPA는 JS가 콘텐츠를 렌더링하는 시간이 필요하므로 충분히 기다려야 합니다.

time.sleep(7)  # 7초 대기 (네이버 웹툰 React 렌더링 시간 확보)

# 로딩 확인: 페이지에 실제 콘텐츠가 렌더링되었는지 체크
page_text = driver.page_source
if "이달의 신규 웹툰" in page_text:
    print("✅ 페이지 로딩 완료! ('이달의 신규 웹툰' 섹션 확인됨)")
else:
    print("⚠️ '이달의 신규 웹툰' 섹션이 아직 안 보입니다. 추가 대기 중...")
    time.sleep(5)  # 5초 더 대기
    if "이달의 신규 웹툰" in driver.page_source:
        print("✅ 추가 대기 후 로딩 완료!")
    else:
        print("⚠️ 콘텐츠를 찾지 못했습니다. 페이지 구조가 변경되었을 수 있습니다.")

✅ 페이지 로딩 완료! ('이달의 신규 웹툰' 섹션 확인됨)


---
## 7. "이달의 신규 웹툰" 섹션 찾기

네이버 웹툰 페이지 구조를 보면, **"이달의 신규 웹툰"** 텍스트를 포함한 헤더 아래에 웹툰 목록이 있습니다.

React/CSS Module 특성상 클래스명 끝에 해시값(예: `--tAZvu`)이 붙어 있으므로,
`[class*='...']` 형태의 **부분 매칭 CSS 셀렉터**를 사용합니다.

> ⚠️ **주의**: 이달의 신규 웹툰은 무작위로 추출되므로, 실행할 때마다 결과가 달라질 수 있습니다.

In [5]:
# 페이지 소스를 BeautifulSoup으로 파싱 (Selenium이 JS 실행 후의 완성된 HTML)
soup = BeautifulSoup(driver.page_source, "html.parser")

# "이달의 신규 웹툰" 텍스트가 포함된 섹션 헤더 찾기
new_webtoon_header = soup.find(string=lambda t: t and "이달의 신규 웹툰" in t)

if new_webtoon_header:
    print(f"✅ 섹션 발견: '{new_webtoon_header.strip()}'")
else:
    print("⚠️ '이달의 신규 웹툰' 헤더를 찾지 못했습니다.")
    print("   → 페이지 구조가 변경되었을 수 있습니다.")

✅ 섹션 발견: '이달의 신규 웹툰'


---
## 8. 이달의 신규 웹툰 데이터 추출

찾은 섹션에서 각 웹툰의 **제목, 작가, 내용(줄거리)**을 추출합니다.

추출 구조:
```
이달의 신규 웹툰 섹션
  └─ 웹툰 아이템 (li)
       ├─ 제목 (strong / span[class*='title'])
       ├─ 작가 (span[class*='author'] / span[class*='Artist'])
       └─ 내용 (p[class*='summary'] / span[class*='summary'])
```

In [6]:
print("=" * 60)
print("📚 이달의 신규 웹툰 목록")
print("=" * 60)

# "이달의 신규 웹툰" 섹션의 웹툰 리스트 (ul) 안의 li 요소들
webtoon_items = driver.find_elements(
    By.CSS_SELECTOR,
    "ul[class*='TripleRecommendList__triple_recommend_list'] > li"
)

if webtoon_items:
    print(f"✅ {len(webtoon_items)}개 항목 발견\n")

    for index, item in enumerate(webtoon_items, start=1):
        # 제목: span.text 안의 텍스트
        try:
            title = item.find_element(
                By.CSS_SELECTOR, "span[class*='ContentTitle__title'] span.text"
            ).text
        except:
            title = "(제목 없음)"

        # 작가: a 태그 중 author 클래스
        try:
            author = item.find_element(
                By.CSS_SELECTOR, "a[class*='author']"
            ).text
        except:
            author = "(작가 없음)"

        # 내용(줄거리): p 태그 중 summary 클래스
        try:
            summary = item.find_element(
                By.CSS_SELECTOR, "p[class*='summary']"
            ).text
        except:
            summary = "(내용 없음)"

        print(f"📖 [{index}]")
        print(f"   제목 : {title}")
        print(f"   작가 : {author}")
        print(f"   내용 : {summary}")
        print(f"{'─' * 60}")

    print(f"\n✅ 총 {len(webtoon_items)}개 웹툰 크롤링 완료")
else:
    print("⚠️ 웹툰 항목을 찾지 못했습니다.")

📚 이달의 신규 웹툰 목록
✅ 3개 항목 발견

📖 [1]
   제목 : 나랑 한판
   작가 : MIMIC
   내용 : 폼생폼사! 싸움 하나만큼은 누구보다 자신 있는 고등학생 박기현. 전학 간 새 학교에서도 당연히 중심이 될 줄 알았지만— “어라, 한방에 보내는 건 내 주특기였는데...?” 결과는 단 한판! 압도적인 상대 앞에서 기현의 자존심은 산산조각 나고, 이기기 위한 일념 하나로 그는 결국 매트 위에 올라서게 되는데— 야, 나랑 한판 하자!!
────────────────────────────────────────────────────────────
📖 [2]
   제목 : 불로불사로 산다는 것은
   작가 : 차원 / 로비 / 겨울에어컨
   내용 : 공교회의 사제 로베르타는 북쪽 끝에 있는 벽지, 디트마르셴의 주임 사제로 발령받는다. 전임자가 실종된 사건이 벌어진 직후 주어진 자리이기에, 로베르타는 경계를 품고 디트마르셴 공작 ‘울리히’에 관해 알아간다. 소문에 의하면 그는 300년 이상 영주로서 자리하고 있는 사람이요, 스스로 말하기를 인간으로서 누릴 수 없는 삶을 산 영생자라니… 로베르타는 진실을 밝히기 위해 그의 행적을 따라가기로 한다.
────────────────────────────────────────────────────────────
📖 [3]
   제목 : 너의 세계
   작가 : 이에내
   내용 : 다정하고 유능한 1학년 2반 반장 허윤아는 담임선생님으로부터 전학생 이도를 챙겨 달라는 부탁을 받는다. 그러나 너무나 다른 두 사람. 상냥하지만 위선적인 반장 허윤아와 고고하지만 솔직한 교포 전학생 이도는 서로를 이해할 수 있을까?
────────────────────────────────────────────────────────────

✅ 총 3개 웹툰 크롤링 완료


---
## 9. 대체 방법: 섹션 전체 텍스트 출력

CSS 셀렉터가 정확히 매칭되지 않을 경우를 대비하여,
**"이달의 신규 웹툰"이 포함된 섹션 전체의 텍스트**를 출력하는 방법입니다.

이 방법은 구조 파악이 어려울 때 **디버깅 용도**로도 유용합니다.

In [7]:
# "이달의 신규 웹툰" 파싱 (신작 라벨 기준으로 분리)
print("=" * 60)
print("📋 이달의 신규 웹툰 결과")
print("=" * 60)

try:
    sections = driver.find_elements(
        By.CSS_SELECTOR,
        "div[class*='component'], div[class*='Component'], section"
    )

    found = False
    for section in sections:
        section_text = section.text
        if "이달의 신규 웹툰" in section_text:
            lines = section_text.strip().split("\n")

            # "신작"을 구분자로 웹툰 그룹 나누기
            groups = []
            current = []
            for line in lines:
                line = line.strip()
                if not line or "이달의 신규 웹툰" in line or "더보기" in line:
                    continue
                if line == "신작":
                    if current:
                        groups.append(current)
                    current = []
                else:
                    current.append(line)
            if current:
                groups.append(current)

            # 각 그룹에서 제목/작가/내용 분류
            for i, group in enumerate(groups, start=1):
                title = ""
                author = ""
                summary = ""

                for item in group:
                    if len(item) > 30:
                        # 30자 이상이면 줄거리(내용)
                        summary = item
                    elif not title:
                        # 첫 번째 짧은 텍스트 = 제목
                        title = item
                    else:
                        # 두 번째 짧은 텍스트 = 작가
                        author = item

                print(f"\n📖 [{i}]")
                print(f"   제목 : {title}")
                print(f"   작가 : {author}")
                print(f"   내용 : {summary}")
                print(f"{'─' * 60}")

            print(f"\n✅ 총 {len(groups)}개 웹툰 발견")
            found = True
            break

    if not found:
        print("⚠️ '이달의 신규 웹툰' 텍스트를 포함한 섹션을 찾지 못했습니다.")

except Exception as e:
    print(f"오류 발생: {e}")

📋 이달의 신규 웹툰 결과

📖 [1]
   제목 : 나랑 한판
   작가 : MIMIC
   내용 : 폼생폼사! 싸움 하나만큼은 누구보다 자신 있는 고등학생 박기현. 전학 간 새 학교에서도 당연히 중심이 될 줄 알았지만— “어라, 한방에 보내는 건 내 주특기였는데...?” 결과는 단 한판! 압도적인 상대 앞에서 기현의 자존심은 산산조각 나고, 이기기 위한 일념 하나로 그는 결국 매트 위에 올라서게 되는데— 야, 나랑 한판 하자!!
────────────────────────────────────────────────────────────

📖 [2]
   제목 : 불로불사로 산다는 것은
   작가 : 차원 / 로비 / 겨울에어컨
   내용 : 공교회의 사제 로베르타는 북쪽 끝에 있는 벽지, 디트마르셴의 주임 사제로 발령받는다. 전임자가 실종된 사건이 벌어진 직후 주어진 자리이기에, 로베르타는 경계를 품고 디트마르셴 공작 ‘울리히’에 관해 알아간다. 소문에 의하면 그는 300년 이상 영주로서 자리하고 있는 사람이요, 스스로 말하기를 인간으로서 누릴 수 없는 삶을 산 영생자라니… 로베르타는 진실을 밝히기 위해 그의 행적을 따라가기로 한다.
────────────────────────────────────────────────────────────

📖 [3]
   제목 : 너의 세계
   작가 : 이에내
   내용 : 다정하고 유능한 1학년 2반 반장 허윤아는 담임선생님으로부터 전학생 이도를 챙겨 달라는 부탁을 받는다. 그러나 너무나 다른 두 사람. 상냥하지만 위선적인 반장 허윤아와 고고하지만 솔직한 교포 전학생 이도는 서로를 이해할 수 있을까?
────────────────────────────────────────────────────────────

✅ 총 3개 웹툰 발견


---
## 10. 브라우저 종료

크롤링이 완료되었으므로 열어둔 브라우저를 종료합니다.

> `driver.quit()`을 호출하지 않으면 백그라운드에서 Chrome 프로세스가 계속 남아 메모리를 차지합니다.

In [8]:
# 브라우저 종료
driver.quit()
print("✅ 브라우저가 정상적으로 종료되었습니다.")

✅ 브라우저가 정상적으로 종료되었습니다.


---
## 정리

| 단계 | 내용 |
|------|------|
| 환경 설정 | `google-colab-selenium` 패키지로 자동 설정 |
| 페이지 접속 | `driver.get(url)` → JS 실행까지 대기 |
| 데이터 추출 | CSS 셀렉터로 제목/작가/내용 추출 |
| 핵심 포인트 | React SPA → BeautifulSoup(정적)은 `None` 반환 → **Selenium(동적) 필수** |

### 주의사항
- 네이버 웹툰의 클래스명에는 해시값(`--tAZvu` 등)이 포함되어 있어, 배포 시 변경될 수 있습니다.
- 이달의 신규 웹툰은 **무작위로 추출**되므로 실행할 때마다 결과가 다를 수 있습니다.
- 과도한 크롤링은 서버에 부담을 주므로 `time.sleep()`으로 적절한 간격을 두는 것이 좋습니다.